# 📊 Code 1d: Fetch Commodities Data

## Purpose
Fetch **Close price** data for common commodities from Yahoo Finance.

## Date Range
**Start Date:** 01-Jul-2009  
**End Date:** Today

## Output
- **`commodities_raw_all.csv`** - All commodities data
- `commodities_not_found.csv` - Commodities not available or with data quality issues

## Commodities to Fetch:
### Precious Metals:
- Gold, Silver, Platinum, Palladium

### Industrial Metals:
- Copper, Aluminium, Steel

### Energy:
- Crude Oil (WTI), Brent Crude Oil, Natural Gas

### Agricultural:
- Sugar, Cotton, Soybean, Wheat, Corn

### Currency / FX:
- USD/INR FX Rate (important for Indian markets — affects IT/Pharma exporters and importers)

---

**⏱️ Estimated Time:** 2-3 minutes

## Step 1: Install and Import Libraries

In [1]:
# Install required packages
!pip install yfinance --quiet
!pip install pandas --quiet

print("✅ Packages installed successfully!")

✅ Packages installed successfully!


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print(f"📅 Script run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully!
📅 Script run date: 2026-06-15 18:33:06


## Step 2: Configuration

In [3]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Output files
OUTPUT_FILE = 'commodities_raw_all.csv'
NOT_FOUND_FILE = 'commodities_not_found.csv'

# Date range
START_DATE = '2007-01-01'
END_DATE = datetime.now().strftime('%Y-%m-%d')

# Commodities with their Yahoo Finance symbols
COMMODITIES = {
    # Precious Metals
    'GOLD': 'GC=F',           # Gold Futures
    'SILVER': 'SI=F',         # Silver Futures
    'PLATINUM': 'PL=F',       # Platinum Futures
    'PALLADIUM': 'PA=F',      # Palladium Futures

    # Industrial Metals
    'COPPER': 'HG=F',         # Copper Futures
    'ALUMINIUM': 'ALI=F',     # Aluminium Futures (may not be available)

    # Energy
    'CRUDE_OIL_WTI': 'CL=F',  # WTI Crude Oil Futures
    'BRENT_CRUDE': 'BZ=F',    # Brent Crude Oil Futures
    'NATURAL_GAS': 'NG=F',    # Natural Gas Futures

    # Agricultural
    'SUGAR': 'SB=F',          # Sugar Futures
    'COTTON': 'CT=F',         # Cotton Futures
    'SOYBEAN': 'ZS=F',        # Soybean Futures
    'WHEAT': 'ZW=F',          # Wheat Futures
    'CORN': 'ZC=F',           # Corn Futures

    # Additional commodities
    'COFFEE': 'KC=F',         # Coffee Futures
    'COCOA': 'CC=F',          # Cocoa Futures

    # Currency / FX
    'USDINR': 'INR=X',        # USD/INR FX Rate
}

print("="*80)
print("CODE 1d: FETCH COMMODITIES DATA")
print("="*80)
print(f"Output File: {OUTPUT_FILE}")
print(f"Date Range: {START_DATE} to {END_DATE}")
print(f"Total commodities to fetch: {len(COMMODITIES)}")
print("\nCommodities:")
print("\nPrecious Metals:")
for name, symbol in list(COMMODITIES.items())[:4]:
    print(f"  - {name}: {symbol}")
print("\nIndustrial Metals:")
for name, symbol in list(COMMODITIES.items())[4:6]:
    print(f"  - {name}: {symbol}")
print("\nEnergy:")
for name, symbol in list(COMMODITIES.items())[6:9]:
    print(f"  - {name}: {symbol}")
print("\nAgricultural:")
for name, symbol in list(COMMODITIES.items())[9:14]:
    print(f"  - {name}: {symbol}")
print("\nAdditional:")
for name, symbol in list(COMMODITIES.items())[14:16]:
    print(f"  - {name}: {symbol}")
print("\nCurrency / FX:")
for name, symbol in list(COMMODITIES.items())[16:]:
    print(f"  - {name}: {symbol}")
print("="*80)

CODE 1d: FETCH COMMODITIES DATA
Output File: commodities_raw_all.csv
Date Range: 2007-01-01 to 2026-06-15
Total commodities to fetch: 17

Commodities:

Precious Metals:
  - GOLD: GC=F
  - SILVER: SI=F
  - PLATINUM: PL=F
  - PALLADIUM: PA=F

Industrial Metals:
  - COPPER: HG=F
  - ALUMINIUM: ALI=F

Energy:
  - CRUDE_OIL_WTI: CL=F
  - BRENT_CRUDE: BZ=F
  - NATURAL_GAS: NG=F

Agricultural:
  - SUGAR: SB=F
  - COTTON: CT=F
  - SOYBEAN: ZS=F
  - WHEAT: ZW=F
  - CORN: ZC=F

Additional:
  - COFFEE: KC=F
  - COCOA: CC=F

Currency / FX:
  - USDINR: INR=X


## Step 3: Download Commodities Data

Download Close price for all commodities.

In [4]:
print("="*80)
print("DOWNLOADING COMMODITIES DATA")
print("="*80)
print(f"Date range: {START_DATE} to {END_DATE}")
print("\n⏳ Starting download...\n")

# Storage for data
all_commodities_data = []
not_found_commodities = []
data_quality_issues = []

# Download each commodity
for commodity_name, yahoo_symbol in COMMODITIES.items():
    print(f"\nFetching {commodity_name} ({yahoo_symbol})...")

    try:
        # Download data
        data = yf.download(
            yahoo_symbol,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,  # Get Close price only
            progress=False
        )

        if not data.empty and len(data) > 0:
            # Successfully downloaded
            data = data.reset_index()

            # Keep only Date and Close
            if 'Close' in data.columns:
                data = data[['Date', 'Close']].copy()
                data.columns = ['Date', 'Close']
            else:
                # Sometimes column name might be different
                data = data[['Date']].copy()
                data['Close'] = data.iloc[:, 1] if len(data.columns) > 1 else np.nan

            # Add commodity name
            data['Commodity'] = commodity_name

            # Check data quality
            missing_pct = data['Close'].isnull().sum() / len(data) * 100

            # Check date range
            first_date = data['Date'].min()
            last_date = data['Date'].max()

            # Store data
            all_commodities_data.append(data)

            print(f"  ✅ Success: {len(data)} records")
            print(f"     Date range: {first_date.date()} to {last_date.date()}")
            print(f"     Missing data: {missing_pct:.2f}%")
            print(f"     Latest price: ${data['Close'].iloc[-1]:.2f}")

            # Flag if data quality issues
            if first_date > pd.Timestamp(START_DATE) + pd.Timedelta(days=365):
                data_quality_issues.append({
                    'Commodity': commodity_name,
                    'Yahoo_Symbol': yahoo_symbol,
                    'Issue': f'Data starts from {first_date.date()} (later than {START_DATE})',
                    'First_Date': first_date,
                    'Last_Date': last_date,
                    'Total_Records': len(data)
                })
                print(f"     ⚠️  Warning: Data starts later than {START_DATE}")

            if missing_pct > 5:
                data_quality_issues.append({
                    'Commodity': commodity_name,
                    'Yahoo_Symbol': yahoo_symbol,
                    'Issue': f'High missing data: {missing_pct:.2f}%',
                    'First_Date': first_date,
                    'Last_Date': last_date,
                    'Total_Records': len(data)
                })
                print(f"     ⚠️  Warning: High missing data percentage")

        else:
            not_found_commodities.append({
                'Commodity': commodity_name,
                'Yahoo_Symbol': yahoo_symbol,
                'Reason': 'No data available'
            })
            print(f"  ❌ No data available")

    except Exception as e:
        not_found_commodities.append({
            'Commodity': commodity_name,
            'Yahoo_Symbol': yahoo_symbol,
            'Reason': str(e)[:100]
        })
        print(f"  ❌ Error: {str(e)[:100]}")

print("\n" + "="*80)
print("✅ DOWNLOAD COMPLETED!")
print("="*80)
print(f"Successfully downloaded: {len(all_commodities_data)} commodities")
print(f"Not found: {len(not_found_commodities)} commodities")
print(f"Data quality issues: {len(data_quality_issues)} commodities")

DOWNLOADING COMMODITIES DATA
Date range: 2007-01-01 to 2026-06-15

⏳ Starting download...


Fetching GOLD (GC=F)...
  ✅ Success: 4892 records
     Date range: 2007-01-02 to 2026-06-12
     Missing data: 0.00%
     Latest price: $4215.00

Fetching SILVER (SI=F)...
  ✅ Success: 4892 records
     Date range: 2007-01-02 to 2026-06-12
     Missing data: 0.00%
     Latest price: $67.86

Fetching PLATINUM (PL=F)...
  ✅ Success: 4484 records
     Date range: 2007-01-02 to 2026-06-12
     Missing data: 0.00%
     Latest price: $1709.20

Fetching PALLADIUM (PA=F)...
  ✅ Success: 4537 records
     Date range: 2007-01-30 to 2026-06-12
     Missing data: 0.00%
     Latest price: $1276.20

Fetching COPPER (HG=F)...
  ✅ Success: 4894 records
     Date range: 2007-01-01 to 2026-06-12
     Missing data: 0.00%
     Latest price: $6.43

Fetching ALUMINIUM (ALI=F)...
  ✅ Success: 3008 records
     Date range: 2014-05-06 to 2026-06-12
     Missing data: 0.00%
     Latest price: $3858.75
     ⚠️  Warning: D

## Step 4: Combine Data into Wide Format

**Important:** Each commodity will be a separate column, with Date as the first column.

In [5]:
if len(all_commodities_data) > 0:
    print("Combining all commodities data into WIDE FORMAT...")
    print("-"*80)

    # Start with first commodity
    df_commodities = all_commodities_data[0][['Date', 'Close']].copy()
    first_commodity = all_commodities_data[0]['Commodity'].iloc[0]
    df_commodities = df_commodities.rename(columns={'Close': first_commodity})

    # Merge remaining commodities
    for i in range(1, len(all_commodities_data)):
        temp_df = all_commodities_data[i][['Date', 'Close']].copy()
        commodity_name = all_commodities_data[i]['Commodity'].iloc[0]
        temp_df = temp_df.rename(columns={'Close': commodity_name})

        # Merge on Date
        df_commodities = df_commodities.merge(temp_df, on='Date', how='outer')

    # Sort by Date
    df_commodities = df_commodities.sort_values('Date').reset_index(drop=True)

    print(f"✅ Combined data in WIDE format!")
    print(f"   Total records (dates): {len(df_commodities):,}")
    print(f"   Total columns: {len(df_commodities.columns)} (1 Date + {len(df_commodities.columns)-1} commodities)")
    print(f"   Date range: {df_commodities['Date'].min()} to {df_commodities['Date'].max()}")
    print(f"\n   Columns: Date, {', '.join(df_commodities.columns[1:])}")

    # Check missing values per commodity
    print(f"\n📊 MISSING VALUES PER COMMODITY:")
    for col in df_commodities.columns[1:]:
        missing = df_commodities[col].isnull().sum()
        missing_pct = missing / len(df_commodities) * 100
        print(f"   {col}: {missing} missing ({missing_pct:.2f}%)")

else:
    print("❌ No data was downloaded successfully!")
    df_commodities = pd.DataFrame()

Combining all commodities data into WIDE FORMAT...
--------------------------------------------------------------------------------
✅ Combined data in WIDE format!
   Total records (dates): 5,070
   Total columns: 18 (1 Date + 17 commodities)
   Date range: 2007-01-01 00:00:00 to 2026-06-12 00:00:00

   Columns: Date, GOLD, SILVER, PLATINUM, PALLADIUM, COPPER, ALUMINIUM, CRUDE_OIL_WTI, BRENT_CRUDE, NATURAL_GAS, SUGAR, COTTON, SOYBEAN, WHEAT, CORN, COFFEE, COCOA, USDINR

📊 MISSING VALUES PER COMMODITY:
   GOLD: 178 missing (3.51%)
   SILVER: 178 missing (3.51%)
   PLATINUM: 586 missing (11.56%)
   PALLADIUM: 533 missing (10.51%)
   COPPER: 176 missing (3.47%)
   ALUMINIUM: 2062 missing (40.67%)
   CRUDE_OIL_WTI: 177 missing (3.49%)
   BRENT_CRUDE: 373 missing (7.36%)
   NATURAL_GAS: 176 missing (3.47%)
   SUGAR: 180 missing (3.55%)
   COTTON: 180 missing (3.55%)
   SOYBEAN: 179 missing (3.53%)
   WHEAT: 179 missing (3.53%)
   CORN: 181 missing (3.57%)
   COFFEE: 181 missing (3.57%)
   C

## Step 5: Handle Missing Values

Apply multiple strategies to handle missing values:
1. Forward fill for small gaps (1-3 days)
2. Linear interpolation for medium gaps (4-10 days)
3. Backward fill for any remaining gaps at the end

This ensures complete data for correlation analysis.

In [6]:
if not df_commodities.empty:
    print("="*80)
    print("HANDLING MISSING VALUES")
    print("="*80)

    # Save raw version first (before preprocessing)
    print(f"\nSaving raw data to commodities_raw_all.csv...")
    df_commodities.to_csv('commodities_raw_all.csv', index=False)
    print(f"✅ Raw data saved")

    # Count missing values before processing
    missing_before = df_commodities.isnull().sum()
    total_missing_before = missing_before.sum()

    print(f"\n📊 BEFORE PROCESSING:")
    print(f"   Total missing values: {total_missing_before:,}")
    if total_missing_before > 0:
        print(f"\n   Missing values by commodity:")
        for col in df_commodities.columns[1:]:
            if missing_before[col] > 0:
                pct = missing_before[col] / len(df_commodities) * 100
                print(f"     {col}: {missing_before[col]} ({pct:.2f}%)")

    # Strategy 1: Forward fill for small gaps (1-3 days)
    print(f"\n🔄 Step 1: Forward fill (limit 3 days)...")
    for col in df_commodities.columns[1:]:
        df_commodities[col] = df_commodities[col].fillna(method='ffill', limit=3)

    missing_after_ffill = df_commodities.isnull().sum().sum()
    filled_ffill = total_missing_before - missing_after_ffill
    print(f"   Filled {filled_ffill} values")
    print(f"   Remaining: {missing_after_ffill}")

    # Strategy 2: Linear interpolation for medium gaps
    print(f"\n🔄 Step 2: Linear interpolation (limit 10 days)...")
    for col in df_commodities.columns[1:]:
        df_commodities[col] = df_commodities[col].interpolate(
            method='linear',
            limit=10,
            limit_direction='both'
        )

    missing_after_interp = df_commodities.isnull().sum().sum()
    filled_interp = missing_after_ffill - missing_after_interp
    print(f"   Filled {filled_interp} values")
    print(f"   Remaining: {missing_after_interp}")

    # Strategy 3: Backward fill for remaining gaps (especially at start/end)
    if missing_after_interp > 0:
        print(f"\n🔄 Step 3: Backward fill (no limit)...")
        for col in df_commodities.columns[1:]:
            df_commodities[col] = df_commodities[col].fillna(method='bfill')

        missing_after_bfill = df_commodities.isnull().sum().sum()
        filled_bfill = missing_after_interp - missing_after_bfill
        print(f"   Filled {filled_bfill} values")
        print(f"   Remaining: {missing_after_bfill}")

    # Final check
    missing_final = df_commodities.isnull().sum()
    total_missing_final = missing_final.sum()

    print(f"\n✅ AFTER PROCESSING:")
    print(f"   Total missing values: {total_missing_final:,}")
    if total_missing_final > 0:
        print(f"\n   ⚠️  Remaining missing values:")
        for col in df_commodities.columns[1:]:
            if missing_final[col] > 0:
                pct = missing_final[col] / len(df_commodities) * 100
                print(f"     {col}: {missing_final[col]} ({pct:.2f}%)")
    else:
        print(f"   🎉 No missing values! Dataset is complete.")

    print(f"\n📊 SUMMARY:")
    print(f"   Total filled: {total_missing_before - total_missing_final:,}")
    print(f"   Fill rate: {(total_missing_before - total_missing_final) / total_missing_before * 100 if total_missing_before > 0 else 100:.2f}%")

    print("-"*80)
else:
    print("⚠️  No data to process")

HANDLING MISSING VALUES

Saving raw data to commodities_raw_all.csv...
✅ Raw data saved

📊 BEFORE PROCESSING:
   Total missing values: 5,729

   Missing values by commodity:
     GOLD: 178 (3.51%)
     SILVER: 178 (3.51%)
     PLATINUM: 586 (11.56%)
     PALLADIUM: 533 (10.51%)
     COPPER: 176 (3.47%)
     ALUMINIUM: 2062 (40.67%)
     CRUDE_OIL_WTI: 177 (3.49%)
     BRENT_CRUDE: 373 (7.36%)
     NATURAL_GAS: 176 (3.47%)
     SUGAR: 180 (3.55%)
     COTTON: 180 (3.55%)
     SOYBEAN: 179 (3.53%)
     WHEAT: 179 (3.53%)
     CORN: 181 (3.57%)
     COFFEE: 181 (3.57%)
     COCOA: 181 (3.57%)
     USDINR: 29 (0.57%)

🔄 Step 1: Forward fill (limit 3 days)...
   Filled 2887 values
   Remaining: 2842

🔄 Step 2: Linear interpolation (limit 10 days)...
   Filled 518 values
   Remaining: 2324

🔄 Step 3: Backward fill (no limit)...
   Filled 2324 values
   Remaining: 0

✅ AFTER PROCESSING:
   Total missing values: 0
   🎉 No missing values! Dataset is complete.

📊 SUMMARY:
   Total filled: 5,729


## Step 6: Save Cleaned Output Files

Save the cleaned/preprocessed data to **commodities_all.csv** (without 'raw' in name).

In [7]:
print("="*80)
print("SAVING CLEANED OUTPUT FILES")
print("="*80)

# Save cleaned commodities data (preprocessed)
CLEANED_OUTPUT_FILE = 'commodities_all.csv'

if not df_commodities.empty:
    print(f"\nSaving cleaned data to {CLEANED_OUTPUT_FILE}...")
    df_commodities.to_csv(CLEANED_OUTPUT_FILE, index=False)
    print(f"✅ Saved: {CLEANED_OUTPUT_FILE}")
    print(f"   Rows: {len(df_commodities):,}")
    print(f"   Columns: {len(df_commodities.columns)} (1 Date + {len(df_commodities.columns)-1} commodities)")
    print(f"   Missing values: {df_commodities.isnull().sum().sum()}")

# Save not found / issues list
if len(not_found_commodities) > 0 or len(data_quality_issues) > 0:
    # Combine not found and quality issues
    all_issues = not_found_commodities + data_quality_issues
    df_issues = pd.DataFrame(all_issues)
    df_issues.to_csv(NOT_FOUND_FILE, index=False)
    print(f"\n📋 Saved: {NOT_FOUND_FILE}")
    print(f"   Commodities with issues: {len(all_issues)}")

print("\n" + "="*80)
print("✅ ALL FILES SAVED!")
print("="*80)
print(f"\n📁 Output Files:")
print(f"   1. commodities_raw_all.csv - Original data (before preprocessing)")
print(f"   2. {CLEANED_OUTPUT_FILE} - Cleaned data (after missing value handling)")
print(f"   3. {NOT_FOUND_FILE} - Issues report")

SAVING CLEANED OUTPUT FILES

Saving cleaned data to commodities_all.csv...
✅ Saved: commodities_all.csv
   Rows: 5,070
   Columns: 18 (1 Date + 17 commodities)
   Missing values: 0

📋 Saved: commodities_not_found.csv
   Commodities with issues: 1

✅ ALL FILES SAVED!

📁 Output Files:
   1. commodities_raw_all.csv - Original data (before preprocessing)
   2. commodities_all.csv - Cleaned data (after missing value handling)
   3. commodities_not_found.csv - Issues report


## Step 6.1: Verify Both Files Created

Check that both raw and cleaned files were created successfully.

In [8]:
import os

print("="*80)
print("FILE VERIFICATION")
print("="*80)

files_to_check = [
    ('commodities_raw_all.csv', 'Raw data (before preprocessing)'),
    ('commodities_all.csv', 'Cleaned data (after preprocessing)'),
    ('commodities_not_found.csv', 'Issues report')
]

print("\nChecking output files:")
for filename, description in files_to_check:
    if os.path.exists(filename):
        size = os.path.getsize(filename) / 1024  # KB
        print(f"\n✅ {filename}")
        print(f"   {description}")
        print(f"   Size: {size:.2f} KB")

        # Load and show info
        try:
            df_check = pd.read_csv(filename)
            print(f"   Rows: {len(df_check):,}")
            print(f"   Columns: {len(df_check.columns)}")
            print(f"   Missing values: {df_check.isnull().sum().sum()}")
        except:
            pass
    else:
        print(f"\n❌ {filename} - NOT FOUND")
        print(f"   {description}")

print("\n" + "="*80)

FILE VERIFICATION

Checking output files:

✅ commodities_raw_all.csv
   Raw data (before preprocessing)
   Size: 1190.34 KB
   Rows: 5,070
   Columns: 18
   Missing values: 5729

✅ commodities_all.csv
   Cleaned data (after preprocessing)
   Size: 1256.12 KB
   Rows: 5,070
   Columns: 18
   Missing values: 0

✅ commodities_not_found.csv
   Issues report
   Size: 0.16 KB
   Rows: 1
   Columns: 6
   Missing values: 0



## Step 7: Display Summary

In [9]:
print("="*80)
print("FINAL SUMMARY - CODE 1d")
print("="*80)

print(f"""
📊 Download Statistics:
   - Total commodities attempted: {len(COMMODITIES)}
   - Successfully downloaded: {len(all_commodities_data)}
   - Not found: {len(not_found_commodities)}
   - Data quality issues: {len(data_quality_issues)}

📈 Output Datasets - WIDE FORMAT:

   1️⃣  commodities_raw_all.csv (RAW - before preprocessing):
      - Total records (dates): {len(df_commodities):,}
      - Total columns: {len(df_commodities.columns) if not df_commodities.empty else 0} (1 Date + {len(df_commodities.columns)-1 if not df_commodities.empty else 0} commodities)
      - Date range: {df_commodities['Date'].min() if not df_commodities.empty else 'N/A'} to {df_commodities['Date'].max() if not df_commodities.empty else 'N/A'}

   2️⃣  commodities_all.csv (CLEANED - after preprocessing):
      - Missing values handled using:
        * Forward fill (1-3 day gaps)
        * Linear interpolation (4-10 day gaps)
        * Backward fill (remaining gaps)
      - Final missing values: {df_commodities.isnull().sum().sum() if not df_commodities.empty else 0}
      - Format: Each commodity is a SEPARATE COLUMN
      - Ready for analysis and modeling

📁 Output Files:
   1. commodities_raw_all.csv - Original data
   2. commodities_all.csv - Cleaned/preprocessed data ⭐
   3. commodities_not_found.csv - Issues and not found commodities

➡️  Use commodities_all.csv for:
   - Commodity correlation analysis
   - Macro-economic factor modeling
   - Inflation hedge analysis
""")

print("="*80)

FINAL SUMMARY - CODE 1d

📊 Download Statistics:
   - Total commodities attempted: 17
   - Successfully downloaded: 17
   - Not found: 0
   - Data quality issues: 1

📈 Output Datasets - WIDE FORMAT:

   1️⃣  commodities_raw_all.csv (RAW - before preprocessing):
      - Total records (dates): 5,070
      - Total columns: 18 (1 Date + 17 commodities)
      - Date range: 2007-01-01 00:00:00 to 2026-06-12 00:00:00

   2️⃣  commodities_all.csv (CLEANED - after preprocessing):
      - Missing values handled using:
        * Forward fill (1-3 day gaps)
        * Linear interpolation (4-10 day gaps)
        * Backward fill (remaining gaps)
      - Final missing values: 0
      - Format: Each commodity is a SEPARATE COLUMN
      - Ready for analysis and modeling

📁 Output Files:
   1. commodities_raw_all.csv - Original data
   2. commodities_all.csv - Cleaned/preprocessed data ⭐
   3. commodities_not_found.csv - Issues and not found commodities

➡️  Use commodities_all.csv for:
   - Commodity co

## Step 7: Display Issues

In [10]:
if len(not_found_commodities) > 0:
    print("\n📋 COMMODITIES NOT FOUND:")
    print("="*80)
    df_not_found = pd.DataFrame(not_found_commodities)
    display(df_not_found)
    print("\n💡 Note: Some commodities may not have futures contracts on Yahoo Finance")
    print("   Alternative sources: MCX India, LME, CME for these commodities")

if len(data_quality_issues) > 0:
    print("\n⚠️  DATA QUALITY ISSUES:")
    print("="*80)
    df_quality = pd.DataFrame(data_quality_issues)
    display(df_quality)


⚠️  DATA QUALITY ISSUES:


,Commodity,Yahoo_Symbol,Issue,First_Date,Last_Date,Total_Records
0,ALUMINIUM,ALI=F,Data starts from 2014-05-06 (later than 2007-0...,2014-05-06,2026-06-12,3008


## Step 8: Preview Data

In [11]:
if not df_commodities.empty:
    print("\nSAMPLE DATA (First 10 rows - WIDE FORMAT):")
    print("="*80)
    print("Note: Each commodity is a separate column")
    display(df_commodities.head(10))

    print("\nLAST 10 ROWS (Recent data):")
    print("="*80)
    display(df_commodities.tail(10))

    print("\nSUMMARY STATISTICS (All Commodities):")
    print("="*80)
    display(df_commodities.describe())

    print("\nLATEST PRICES (Most recent date):")
    print("="*80)
    latest_row = df_commodities.iloc[-1]
    print(f"Date: {latest_row['Date']}")
    print("\nPrices:")
    for col in df_commodities.columns[1:]:
        if pd.notna(latest_row[col]):
            print(f"  {col}: ${latest_row[col]:.2f}")


SAMPLE DATA (First 10 rows - WIDE FORMAT):
Note: Each commodity is a separate column


,Date,GOLD,SILVER,PLATINUM,PALLADIUM,COPPER,ALUMINIUM,CRUDE_OIL_WTI,BRENT_CRUDE,NATURAL_GAS,SUGAR,COTTON,SOYBEAN,WHEAT,CORN,COFFEE,COCOA,USDINR
0,2007-01-01,635.200012,12.818,1139.300049,341.25,2.8540,2172.75,61.049999,75.739998,6.299,11.51,54.889999,669.25,476.50,370.50,123.599998,1684.0,44.215000
1,2007-01-02,635.200012,12.818,1139.300049,341.25,2.8540,2172.75,61.049999,75.739998,6.299,11.51,54.889999,669.25,476.50,370.50,123.599998,1684.0,44.122002
2,2007-01-03,627.099976,12.555,1132.400024,341.25,2.6325,2172.75,58.320000,75.739998,6.163,11.51,54.889999,669.25,476.50,370.50,123.599998,1684.0,44.111000
3,2007-01-04,623.900024,12.727,1132.500000,341.25,2.5885,2172.75,55.590000,75.739998,6.162,11.27,54.560001,662.75,467.50,362.25,125.000000,1659.0,44.095001
4,2007-01-05,604.900024,12.130,1109.000000,341.25,2.5225,2172.75,56.310001,75.739998,6.184,11.09,54.419998,668.00,470.25,368.25,120.449997,1607.0,44.139999
5,2007-01-08,607.500000,12.260,1119.400024,341.25,2.5145,2172.75,56.090000,75.739998,6.378,11.16,54.529999,665.00,464.00,363.50,120.099998,1589.0,44.255001
6,2007-01-09,613.099976,12.498,1127.699951,341.25,2.5420,2172.75,55.639999,75.739998,6.631,11.10,54.099998,653.50,453.00,354.50,118.400002,1588.0,44.209999
7,2007-01-10,611.599976,12.350,1150.800049,341.25,2.6515,2172.75,54.020000,75.739998,6.755,11.12,53.930000,653.75,449.50,360.25,120.750000,1617.0,44.375000
8,2007-01-11,612.400024,12.375,1138.800049,341.25,2.6485,2172.75,51.880001,75.739998,6.292,11.02,54.040001,664.00,456.50,376.50,120.650002,1628.0,44.424999
9,2007-01-12,625.500000,12.798,1146.099976,341.25,2.5930,2172.75,52.990002,75.739998,6.601,10.93,54.700001,706.00,479.50,396.50,120.300003,1631.0,44.255001



LAST 10 ROWS (Recent data):


,Date,GOLD,SILVER,PLATINUM,PALLADIUM,COPPER,ALUMINIUM,CRUDE_OIL_WTI,BRENT_CRUDE,NATURAL_GAS,SUGAR,COTTON,SOYBEAN,WHEAT,CORN,COFFEE,COCOA,USDINR
5060,2026-06-01,4475.200195,75.007004,1922.400024,1360.500000,6.5240,3980.75,92.160004,94.980003,3.179,14.45,76.639999,1180.75,608.75,444.00,260.600006,3895.0,95.002998
5061,2026-06-02,4489.100098,75.310997,1937.400024,1373.099976,6.6495,4054.25,93.760002,96.000000,3.167,14.38,77.040001,1165.25,603.00,440.50,259.200012,4108.0,95.551399
5062,2026-06-03,4436.700195,73.475998,1868.699951,1317.099976,6.4810,4026.25,96.019997,97.809998,3.214,14.24,76.730003,1154.00,587.25,431.50,253.100006,4072.0,95.263000
5063,2026-06-04,4475.799805,73.778999,1894.000000,1318.800049,6.5110,3996.75,93.040001,95.029999,3.336,14.27,74.889999,1129.50,581.75,424.50,247.149994,3965.0,96.164497
5064,2026-06-05,4337.100098,68.943001,1792.000000,1247.099976,6.2635,3950.25,90.540001,93.089996,3.229,14.14,73.750000,1121.50,580.00,417.50,246.500000,3762.0,95.790298
5065,2026-06-08,4335.899902,68.425003,1749.400024,1200.900024,6.3295,3953.50,91.300003,94.250000,3.147,14.12,73.389999,1115.75,583.25,418.75,245.899994,3831.0,94.950302
5066,2026-06-09,4260.000000,65.094002,1708.599976,1213.599976,6.3025,3881.50,88.199997,91.449997,3.140,14.08,71.260002,1113.75,585.25,419.50,244.399994,3831.0,95.688301
5067,2026-06-10,4108.200195,64.598999,1688.000000,1230.800049,6.2490,3779.75,90.029999,93.099998,3.185,13.92,71.099998,1123.00,587.50,419.00,248.399994,3757.0,95.360298
5068,2026-06-11,4090.300049,63.884998,1662.599976,1234.599976,6.2590,3794.50,87.709999,90.379997,3.087,13.79,72.489998,1115.00,586.75,411.75,253.949997,3710.0,95.644501
5069,2026-06-12,4215.000000,67.859001,1709.199951,1276.199951,6.4305,3858.75,84.879997,87.330002,3.120,13.70,72.940002,1113.50,584.50,412.75,257.200012,3779.0,95.760101



SUMMARY STATISTICS (All Commodities):


,Date,GOLD,SILVER,PLATINUM,PALLADIUM,COPPER,ALUMINIUM,CRUDE_OIL_WTI,BRENT_CRUDE,NATURAL_GAS,SUGAR,COTTON,SOYBEAN,WHEAT,CORN,COFFEE,COCOA,USDINR
count,5070,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000,5070.000000
mean,2016-09-18 22:27:58.579881728,1606.704220,23.067758,1216.433514,1027.759540,3.391589,2213.491913,72.816296,78.325102,3.894915,17.181923,78.527716,1146.070217,607.218688,472.097781,169.074931,3207.488363,64.518174
min,2007-01-01 00:00:00,604.900024,8.790000,595.900024,162.100006,1.247500,1452.000000,-37.630001,19.330000,1.482000,8.450000,39.139999,653.500000,361.000000,293.500000,86.650002,1578.000000,39.044998
25%,2011-11-10 06:00:00,1203.925018,16.288250,940.000000,607.562515,2.760875,2172.750000,56.340000,61.939999,2.699250,13.132500,64.492498,954.062500,503.500000,368.750000,120.449997,2330.000000,50.050750
50%,2016-09-19 12:00:00,1352.650024,19.402000,1085.699951,805.425018,3.309750,2172.750000,71.795002,75.739998,3.320000,16.530001,73.245003,1060.500000,567.750000,421.500000,141.375000,2656.500000,65.413799
75%,2021-07-28 18:00:00,1804.950012,26.785751,1470.017055,1358.000000,3.906375,2309.250000,89.675001,97.632500,4.369000,19.840000,84.937502,1354.437500,688.937500,571.375000,195.850006,3088.000000,74.899178
max,2026-06-12 00:00:00,5318.399902,115.080002,2852.399902,2985.399902,6.649500,4195.000000,145.289993,146.080002,13.577000,35.310001,215.149994,1771.000000,1425.250000,831.250000,438.899994,12565.000000,96.565804
std,NaN,774.581976,11.808856,344.214107,628.151370,0.869818,348.265290,21.664099,23.474979,1.899701,5.003575,23.624272,239.594204,147.499414,131.679799,69.237636,1850.003806,14.672227



LATEST PRICES (Most recent date):
Date: 2026-06-12 00:00:00

Prices:
  GOLD: $4215.00
  SILVER: $67.86
  PLATINUM: $1709.20
  PALLADIUM: $1276.20
  COPPER: $6.43
  ALUMINIUM: $3858.75
  CRUDE_OIL_WTI: $84.88
  BRENT_CRUDE: $87.33
  NATURAL_GAS: $3.12
  SUGAR: $13.70
  COTTON: $72.94
  SOYBEAN: $1113.50
  WHEAT: $584.50
  CORN: $412.75
  COFFEE: $257.20
  COCOA: $3779.00
  USDINR: $95.76


## 📥 Download Files (Optional)

In [12]:
from google.colab import files

print("Downloading files...")
files.download(OUTPUT_FILE)
if len(not_found_commodities) > 0 or len(data_quality_issues) > 0:
    files.download(NOT_FOUND_FILE)

files.download("commodities_all.csv")

print("\n✅ Download complete!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!


---

## ✅ Code 1d Complete!

### What This Data Can Be Used For:
- **Commodity correlation analysis** - Which stocks move with gold, oil, etc.
- **Macro-economic modeling** - Commodity prices as predictive features
- **Inflation hedge analysis** - How stocks perform vs commodity inflation
- **Sector correlation** - Energy stocks vs crude oil, metal stocks vs copper

### Notes:
- Some commodities (like steel, aluminium) may not have easily accessible futures data on Yahoo Finance
- For Indian commodity prices, consider MCX (Multi Commodity Exchange) data
- Futures prices shown here are global benchmarks (COMEX, NYMEX, CME)